In [1]:
import os
import cv2
import math
import mediapipe as mp
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
TRAIN_DIR = "../data/alphabet/raw/train"
TEST_DIR = "../data/alphabet/raw/test"

OUTPUT_TRAIN_CSV = "../data/alphabet/landmarks/train_landmarks_normalized.csv"
OUTPUT_TEST_CSV = "../data/alphabet/landmarks/test_landmarks_normalized.csv"

In [ ]:
mp_hands = mp.solutions.hands

In [4]:
def normalize_landmarks(landmarks):
    """
    landmarks: list of (x, y, z) tuples for 21 hand landmarks
    Returns flattened normalized landmark vector
    """
    wrist = landmarks[0]
    
    # Shift so wrist is at origin
    shifted = []
    for x, y, z in landmarks:
        shifted.append((x - wrist[0], y - wrist[1], z - wrist[2]))
    
    # Scale using max distance from wrist
    max_dist = 0.0
    for x, y, z in shifted:
        dist = math.sqrt(x**2 + y**2 + z**2)
        if dist > max_dist:
            max_dist = dist

    if max_dist == 0:
        return None

    normalized = []
    for x, y, z in shifted:
        normalized.extend([x / max_dist, y / max_dist, z / max_dist])

    return normalized

In [ ]:
def extract_landmarks_from_image(image_path, hands):
    image = cv2.imread(image_path)
    if image is None:
        return None

    image = preprocess_image_for_mediapipe(image, pad=80, target_size=512)

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)

    if not results.multi_hand_landmarks:
        return None

    hand_landmarks = results.multi_hand_landmarks[0]

    coords = []
    for lm in hand_landmarks.landmark:
        coords.append((lm.x, lm.y, lm.z))

    normalized = normalize_landmarks(coords)
    return normalized

In [ ]:
def process_dataset(input_dir, split_name):
    data = []
    stats = defaultdict(lambda: {"total": 0, "kept": 0, "failed": 0})

    with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.3
) as hands:

        for label in sorted(os.listdir(input_dir)):
            label_path = os.path.join(input_dir, label)

            if not os.path.isdir(label_path):
                continue

            print(f"Processing {split_name} label: {label}")

            for file_name in os.listdir(label_path):
                file_path = os.path.join(label_path, file_name)
                stats[label]["total"] += 1

                features = extract_landmarks_from_image(file_path, hands)

                if features is not None:
                    row = features + [label, file_path, split_name]
                    data.append(row)
                    stats[label]["kept"] += 1
                else:
                    stats[label]["failed"] += 1

    columns = []
    for i in range(21):
        columns.extend([f"x{i}", f"y{i}", f"z{i}"])
    columns += ["label", "file_path", "split"]

    df = pd.DataFrame(data, columns=columns)
    stats_df = pd.DataFrame(stats).T.reset_index().rename(columns={"index": "label"})

    return df, stats_df

In [7]:
train_df, train_stats = process_dataset(TRAIN_DIR, "train")
test_df, test_stats = process_dataset(TEST_DIR, "test")

Processing train label: A


d:\Fontys\Semester 4 - ML\Sign Language Recognition\Project\venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing train label: B
Processing train label: C
Processing train label: D
Processing train label: E
Processing train label: F
Processing train label: G
Processing train label: H
Processing train label: I
Processing train label: K
Processing train label: L
Processing train label: M
Processing train label: N
Processing train label: O
Processing train label: P
Processing train label: Q
Processing train label: R
Processing train label: S
Processing train label: T
Processing train label: U
Processing train label: V
Processing train label: W
Processing train label: X
Processing train label: Y
Processing test label: A


d:\Fontys\Semester 4 - ML\Sign Language Recognition\Project\venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing test label: B
Processing test label: C
Processing test label: D
Processing test label: E
Processing test label: F
Processing test label: G
Processing test label: H
Processing test label: I
Processing test label: K
Processing test label: L
Processing test label: M
Processing test label: N
Processing test label: O
Processing test label: P
Processing test label: Q
Processing test label: R
Processing test label: S
Processing test label: T
Processing test label: U
Processing test label: V
Processing test label: W
Processing test label: X
Processing test label: Y


In [8]:
os.makedirs("../data/alphabet/landmarks", exist_ok=True)

train_df.to_csv(OUTPUT_TRAIN_CSV, index=False)
test_df.to_csv(OUTPUT_TEST_CSV, index=False)

print("Saved:")
print(OUTPUT_TRAIN_CSV)
print(OUTPUT_TEST_CSV)

Saved:
../data/alphabet/landmarks/train_landmarks_normalized.csv
../data/alphabet/landmarks/test_landmarks_normalized.csv


In [9]:
train_stats["keep_rate"] = train_stats["kept"] / train_stats["total"]
test_stats["keep_rate"] = test_stats["kept"] / test_stats["total"]

train_stats.sort_values("kept", ascending=False)

,label,total,kept,failed,keep_rate
0,A,447,219,228,0.489933
6,G,435,215,220,0.494253
23,Y,438,212,226,0.484018
9,K,455,164,291,0.360440
4,E,441,142,299,0.321995
18,T,414,142,272,0.342995
8,I,433,120,313,0.277136
10,L,423,119,304,0.281324
15,Q,449,98,351,0.218263
14,P,438,97,341,0.221461


In [10]:
test_stats.sort_values("kept", ascending=False)

,label,total,kept,failed,keep_rate
18,T,75,32,43,0.426667
12,N,75,31,44,0.413333
4,E,75,26,49,0.346667
6,G,75,21,54,0.280000
10,L,75,20,55,0.266667
0,A,75,18,57,0.240000
14,P,75,13,62,0.173333
15,Q,75,11,64,0.146667
9,K,75,10,65,0.133333
23,Y,75,9,66,0.120000
